# Week 7 - 01: Retrieval Foundation
This notebook keeps the useful Week 6 retrieval work that Week 7 needs.
We will:
1. Create a small document set.
2. Turn documents into embeddings.
3. Store embeddings in ChromaDB.
4. Search for relevant chunks.

In [ ]:
!pip install chromadb sentence-transformers

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load the embedding model.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create a local ChromaDB database.
db_client = chromadb.PersistentClient(path="./week7_rag_db")

# Create the collection, or open it if it already exists.
collection = db_client.get_or_create_collection(name="documents")

print("Database is ready!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database is ready!


## Step 1: Add our small documents

In [ ]:
documents = [
    "Artificial Intelligence allows computers to perform tasks that normally require human intelligence.",
    "Machine learning allows a computer to learn patterns from examples and data.",
    "Natural language processing helps computers understand and generate human language.",
    "Vector databases store embeddings and support semantic similarity search.",
    "Semantic search retrieves information based on meaning rather than exact keywords.",
    "AI scheduling can assign teachers, rooms, courses, and time slots while following constraints.",
    "Constraint satisfaction problems represent variables, possible values, and constraints.",
    "Genetic algorithms improve solutions through selection, crossover, and mutation."
]

# Give every document a simple ID.
ids = [f"chunk_{i}" for i in range(len(documents))]

# Convert every document into an embedding.
embeddings = embedding_model.encode(documents, convert_to_numpy=True)

# Store the documents and embeddings in ChromaDB.
collection.upsert(                      # upsert means add or update them
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

print("Stored chunks:", collection.count())


Stored chunks: 8


## Step 2: Search the documents

In [ ]:
def search_documents(question, top_k=3):
    # Convert the question into an embedding.
    question_embedding = embedding_model.encode(question).tolist()

    # Ask ChromaDB for the closest matching chunks.
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    # Get the matching document text.
    chunks = results["documents"][0]

    print("\nQuestion:", question)
    print("-" * 60)

    for i, chunk in enumerate(chunks, start=1):
        print(f"Result {i}: {chunk}")

    return chunks


In [ ]:
# Test retrieval before adding the LLM.
search_documents("How can AI create a timetable?")
search_documents("What is semantic search?")
search_documents("How does a genetic algorithm improve a solution?")



Question: How can AI create a timetable?
------------------------------------------------------------
Result 1: AI scheduling can assign teachers, rooms, courses, and time slots while following constraints.
Result 2: Artificial Intelligence allows computers to perform tasks that normally require human intelligence.
Result 3: Machine learning allows a computer to learn patterns from examples and data.

Question: What is semantic search?
------------------------------------------------------------
Result 1: Semantic search retrieves information based on meaning rather than exact keywords.
Result 2: Vector databases store embeddings and support semantic similarity search.
Result 3: Natural language processing helps computers understand and generate human language.

Question: How does a genetic algorithm improve a solution?
------------------------------------------------------------
Result 1: Genetic algorithms improve solutions through selection, crossover, and mutation.
Result 2: Artif

['Genetic algorithms improve solutions through selection, crossover, and mutation.',
 'Artificial Intelligence allows computers to perform tasks that normally require human intelligence.',
 'Machine learning allows a computer to learn patterns from examples and data.']